# Kyrgyz Hate Speech Detection — Submission Notebook

**NLP Course Project, Master's Program**  
**Topic:** Building the first Kyrgyz hate-speech detection dataset and benchmarking classical ML, multilingual transformers, and LLM prompting.

**Contributions:**
1. A 3-class (`hate` / `offensive` / `non_hate`) Kyrgyz hate-speech dataset built from ~13.9k YouTube comments.
2. A benchmark of **8 systems** across three model families.
3. Error analysis tailored to a morphologically rich, low-resource Turkic language.

This notebook is the **final submission artefact**. It runs top to bottom and displays every figure and table the paper uses.  
Heavy training is **not re-run here** — it has already produced files under `results/`, `figures/`, and `tables/`. Re-running everything from scratch is documented in the project README.

In [ ]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))
import pandas as pd, matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display
from src.paths import (
    COMMENTS_FILTERED, CANDIDATES_HATE, CANDIDATES_RANDOM,
    DATASET_FINAL, TRAIN, VAL, TEST, SUMMARY, RESULTS,
    FIGURES, TABLES, DISCOVERIES, LABELS,
)
ROOT = pathlib.Path.cwd().parent
def fig(name):
    p = FIGURES / f'{name}.png'
    return Image(str(p)) if p.exists() else Markdown(f'_figure `{name}` not yet generated_')
def table(name):
    p = TABLES / f'{name}.md'
    return Markdown(p.read_text()) if p.exists() else Markdown(f'_table `{name}` not yet generated_')

## 1. Dataset

### 1.1 Source and filtering
Collected ~15k comments from 13 Kyrgyz-language YouTube videos (political/news domain) via `yt-dlp`. An 11-stage filter (script: `filter_comments.py`) removed empty/short/spam/non-Cyrillic/duplicate comments and produced **13,902 clean comments**.

In [ ]:
df = pd.read_csv(COMMENTS_FILTERED)
print(f'Filtered comments: {len(df):,}')
print(f'Videos: {df["video_id"].nunique()}')
print(f'With Kyrgyz-specific letters (Ң/Ө/Ү): {df["has_kyrgyz_letters"].mean()*100:.1f}%')
fig('pipeline_flowchart')

### 1.2 Candidate sampling (Davidson-style)
We do not annotate all 13,902 comments. Instead, following Davidson et al. (2017) and the Kazakh hate-speech paper, we use a curated Kyrgyz/Russian slur lexicon to extract a *biased* candidate pool (likely-hate), plus a small random pool (likely-non-hate). Both pools are then human-annotated.

In [ ]:
fig('keyword_category_hits')

### 1.3 Annotation
Each candidate was labelled in three classes: `hate` (targets a protected group), `offensive` (profanity/insult without a protected target), `non_hate`. As a second annotator we used **Aya-Expanse-8B** (Cohere) prompted with the same guidelines, and computed Cohen's κ against the human annotator.

In [ ]:
display(fig('annotation_flowchart'))
display(fig('iaa_confusion'))
display(table('annotation_iaa'))

### 1.4 Final dataset statistics

In [ ]:
display(table('dataset_stats'))
display(fig('class_distribution'))
display(fig('text_length_dist'))

## 2. Experimental Setup

Eight systems across three families:

| # | System | Family | Key hyperparameters |
|---|---|---|---|
| 1 | TF-IDF (word 1-2) + LogReg, no preprocessing | classical | min_df=2, balanced |
| 2 | TF-IDF (word 1-2) + LogReg, **with preprocessing** | classical | same, full normalisation |
| 3 | TF-IDF (char 3-5) + LogReg | classical | sublinear TF |
| 4 | TF-IDF (char 3-5) + LinearSVC | classical | C=1.0, calibrated |
| 5 | mBERT fine-tuned | transformer | 5 epochs, bs 16, lr 2e-5 |
| 6 | XLM-RoBERTa-base fine-tuned | transformer | same recipe |
| 7 | Aya-Expanse-8B zero-shot | LLM | greedy, max 8 tokens |
| 8 | Aya-Expanse-8B 5-shot/class | LLM | 15 in-context examples |

Metric: **macro-F1** (primary), per-class F1, accuracy. Test set is stratified 20% of the annotated data.

## 3. Results

In [ ]:
display(fig('results_f1_bar'))
display(table('results_main'))

In [ ]:
display(fig('per_class_metrics'))
display(fig('confusion_matrices'))

### 3.1 RQ1–RQ2 — Preprocessing and character n-grams

In [ ]:
display(table('ablation_preprocessing'))

### 3.2 RQ4–RQ7 — Transformers vs LLMs

In [ ]:
display(table('llm_vs_finetune'))
display(fig('training_curves'))

## 4. Error Analysis

In [ ]:
display(fig('error_categories'))
display(table('error_examples'))

## 5. Discoveries log

A running log of surprising findings throughout this project — kept as raw material for the paper's discussion and for talking to the professor.

In [ ]:
display(Markdown(DISCOVERIES.read_text()))

## 6. Conclusions

Filled by the paper-writing turn. The notebook simply surfaces every artefact the paper depends on.